# GitLab Issues Fetcher - FINAL VERSION

## Column Order:
- title → **labels** → **epic** → **weight** → description → ...

## Features:
1. ✅ Labels, epic, weight after title
2. ✅ linked_project_name
3. ✅ Logging
4. ✅ Archive: CREATE IF NOT EXISTS
5. ✅ Main: DROP and CREATE
6. ✅ Iteration with dates
7. ✅ COPY inserts
8. ✅ Parallel processing

In [ ]:
import requests
import pandas as pd
import getpass
from typing import List, Dict, Tuple
from datetime import datetime
import psycopg2
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils.dataframe import dataframe_to_rows
from concurrent.futures import ThreadPoolExecutor, as_completed
from io import StringIO
import logging
import sys
import warnings
warnings.filterwarnings('ignore')

log_filename = f'gitlab_fetch_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log'
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.FileHandler(log_filename), logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)

logger.info("Notebook initialized")
print(f"✓ Libraries loaded")
print(f"📝 Log: {log_filename}")

In [ ]:
GITLAB_URL = 'https://devcloud.ubs.net'
IKG_PROJECT_PATH = 'ubs/gwma/smart-technology-and-analytics/staat-data-science/staat-ds-insights-cl/commons/staat-ds-insights-home'
SWAT_PROJECT_PATH = 'ubs/gwma/smart-technology-and-analytics/staat-data-science/staat-ds-insights-cl/commons/staat-ds-insights-cl-home'

GREENPLUM_HOST = 'greenplum-rdsp.zur.swissbank.com'
GREENPLUM_PORT = 5432
GREENPLUM_DB = 'gprdsp'
GREENPLUM_USER = 'ds_rdsp_dev'
GREENPLUM_SCHEMA = 'sandbox_prj_smart_insights'

OUTPUT_TABLE1 = 'ikg_issue_details'
OUTPUT_TABLE2 = 'swat_issue_details'
ARCHIVE_TABLE1 = 'ikg_issue_details_archive'
ARCHIVE_TABLE2 = 'swat_issue_details_archive'

MAX_WORKERS = 20
print("✓ Config set")

In [ ]:
gitlab_token = getpass.getpass("GitLab Token: ")
headers = {'PRIVATE-TOKEN': gitlab_token, 'Content-Type': 'application/json'}
project_id_to_name = {}
print("✓ Authenticated")

In [ ]:
def get_project_id(url, path):
    logger.info(f"Fetching: {path}")
    r = requests.get(f"{url}/api/v4/projects/{requests.utils.quote(path, safe='')}", headers=headers)
    r.raise_for_status()
    d = r.json()
    project_id_to_name[str(d['id'])] = d['name']
    logger.info(f"Found: {d['name']} (ID: {d['id']})")
    return d['id'], d['name']

def get_project_name(proj_id):
    pid = str(proj_id)
    if pid in project_id_to_name:
        return project_id_to_name[pid]
    try:
        r = requests.get(f"{GITLAB_URL}/api/v4/projects/{proj_id}", headers=headers, timeout=5)
        r.raise_for_status()
        name = r.json()['name']
        project_id_to_name[pid] = name
        return name
    except:
        return f"Project_{proj_id}"

def fetch_links(pid, iid):
    try:
        r = requests.get(f"{GITLAB_URL}/api/v4/projects/{pid}/issues/{iid}/links", headers=headers, timeout=10)
        r.raise_for_status()
        return r.json()
    except:
        return []

def fetch_links_parallel(pid, iids, workers=20):
    logger.info(f"Fetching links for {len(iids)} issues ({workers} threads)")
    links_map = {}
    with ThreadPoolExecutor(max_workers=workers) as executor:
        futures = {executor.submit(fetch_links, pid, iid): iid for iid in iids}
        for future in as_completed(futures):
            try:
                links_map[futures[future]] = future.result()
            except:
                links_map[futures[future]] = []
    return links_map

def fetch_all_issues(pid, pname, workers=20):
    issues = []
    page = 1
    print(f"\nFetching {pname}...")
    
    while True:
        r = requests.get(f"{GITLAB_URL}/api/v4/projects/{pid}/issues",
                        headers=headers,
                        params={'per_page': 100, 'page': page, 'state': 'all', 'scope': 'all', 'with_labels_details': True})
        r.raise_for_status()
        batch = r.json()
        if not batch:
            break
        issues.extend(batch)
        print(f"  Page {page}: {len(issues)} total")
        if len(batch) < 100:
            break
        page += 1
    
    print(f"  Fetching links...")
    links_map = fetch_links_parallel(pid, [i['iid'] for i in issues], workers)
    for i in issues:
        i['_links_data'] = links_map.get(i['iid'], [])
    
    print(f"✓ {len(issues)} issues with links")
    return issues

print("✓ Functions ready")

In [ ]:
def extract_data(issues, proj):
    logger.info(f"Extracting {len(issues)} issues for {proj}")
    
    if issues and len(issues) > 0:
        first = issues[0]
        if first.get('iteration'):
            logger.info(f"Sample iteration: {first.get('iteration')}")
    
    data = []
    now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    
    for i in issues:
        # Labels
        labels = i.get('labels', [])
        label_str = ', '.join([l.get('name', '') if isinstance(l, dict) else l for l in labels]) if labels else None
        
        # Epic
        epic = i.get('epic', {}).get('title', '') if i.get('epic') else None
        epic_iid = i.get('epic', {}).get('iid', '') if i.get('epic') else None
        
        # Iteration with dates
        iteration = None
        iteration_start_date = None
        iteration_end_date = None
        
        if i.get('iteration'):
            iter_data = i.get('iteration')
            if isinstance(iter_data, dict):
                iteration = (iter_data.get('title') or iter_data.get('name') or 
                           (iter_data.get('web_url', '').split('/')[-1] if iter_data.get('web_url') else None))
                iteration_start_date = iter_data.get('start_date')
                iteration_end_date = iter_data.get('due_date') or iter_data.get('end_date')
            elif isinstance(iter_data, str):
                iteration = iter_data
            
            if iteration:
                logger.debug(f"Issue {i.get('iid')} iteration: {iteration} ({iteration_start_date} to {iteration_end_date})")
        
        # Links
        links = i.get('_links_data', [])
        link_proj_ids = [str(l.get('project_id', '')) for l in links if l.get('project_id')]
        link_proj_names = [get_project_name(pid) for pid in link_proj_ids]
        
        # Column order: project, issue_id, issue_iid, title, labels, epic, weight, description, ...
        data.append({
            'project': proj,
            'issue_id': i.get('id'),
            'issue_iid': i.get('iid'),
            'title': i.get('title'),
            'labels': label_str,
            'epic': epic,
            'weight': i.get('weight'),
            'description': i.get('description'),
            'state': i.get('state'),
            'web_url': i.get('web_url', ''),
            'link_id': ', '.join([str(l.get('id', '')) for l in links if l.get('id')]) or None,
            'link_issue_id': ', '.join([str(l.get('issue_link_id', '')) for l in links if l.get('issue_link_id')]) or None,
            'link_issue_iid': ', '.join([str(l.get('iid', '')) for l in links if l.get('iid')]) or None,
            'link_type': ', '.join([str(l.get('link_type', '')) for l in links if l.get('link_type')]) or None,
            'link_url': ', '.join([str(l.get('web_url', '')) for l in links if l.get('web_url')]) or None,
            'link_issue_title': ', '.join([str(l.get('title', '')) for l in links if l.get('title')]) or None,
            'linked_project_id': ', '.join(link_proj_ids) or None,
            'linked_project_name': ', '.join(link_proj_names) or None,
            'author': i.get('author', {}).get('name'),
            'author_username': i.get('author', {}).get('username'),
            'created_by_id': i.get('author', {}).get('id'),
            'assignee': ', '.join([a.get('name', '') for a in i.get('assignees', [])]) or None,
            'assignee_ids': ', '.join([str(a.get('id', '')) for a in i.get('assignees', [])]),
            'issue_created_date': i.get('created_at'),
            'created_at': i.get('created_at'),
            'updated_at': i.get('updated_at'),
            'closed_at': i.get('closed_at'),
            'due_date': i.get('due_date'),
            'start_date': i.get('start_date'),
            'current_date_time': now,
            'milestone': i.get('milestone', {}).get('title', '') if i.get('milestone') else None,
            'iteration': iteration,
            'iteration_start_date': iteration_start_date,
            'iteration_end_date': iteration_end_date,
            'epic_iid': epic_iid,
            'parent_iid': None,
            'has_tasks': i.get('has_tasks'),
            'task_completion_status': f"{i['task_completion_status'].get('completed_count', 0)}/{i['task_completion_status'].get('count', 0)}" if i.get('task_completion_status') else None,
            'participants': ', '.join([p.get('name', '') for p in i.get('participants', [])]) or None,
            'upvotes': i.get('upvotes'),
            'downvotes': i.get('downvotes'),
            'user_notes_count': i.get('user_notes_count'),
            'merge_requests_count': i.get('merge_requests_count'),
            'time_estimate_hours': i.get('time_stats', {}).get('time_estimate') / 3600 if i.get('time_stats', {}).get('time_estimate') else None,
            'time_spent_hours': i.get('time_stats', {}).get('total_time_spent') / 3600 if i.get('time_stats', {}).get('total_time_spent') else None,
            'confidential': i.get('confidential'),
            'discussion_locked': i.get('discussion_locked'),
            'issue_type': i.get('issue_type'),
            'severity': i.get('severity'),
            'health_status': i.get('health_status'),
        })
    
    df = pd.DataFrame(data)
    
    # Log statistics
    iter_count = df['iteration'].notna().sum()
    iter_dates = df[(df['iteration_start_date'].notna()) | (df['iteration_end_date'].notna())].shape[0]
    logger.info(f"Extracted {len(df)} rows, {len(df.columns)} columns")
    logger.info(f"Issues with iteration: {iter_count}/{len(df)}")
    logger.info(f"Issues with iteration dates: {iter_dates}/{len(df)}")
    if iter_count > 0:
        logger.info(f"Unique iterations: {df['iteration'].unique()[:10]}")
    
    return df

print("✓ Extract ready")

In [ ]:
# Fetch IKG
ikg_id, ikg_name = get_project_id(GITLAB_URL, IKG_PROJECT_PATH)
ikg_issues = fetch_all_issues(ikg_id, ikg_name, MAX_WORKERS)
ikg_df = extract_data(ikg_issues, 'staat-ds-insights-home')
print(f"✓ IKG: {len(ikg_df)} rows")

In [ ]:
# Fetch SWAT
swat_id, swat_name = get_project_id(GITLAB_URL, SWAT_PROJECT_PATH)
swat_issues = fetch_all_issues(swat_id, swat_name, MAX_WORKERS)
swat_df = extract_data(swat_issues, 'staat-ds-insights-cl-home')
print(f"✓ SWAT: {len(swat_df)} rows")

In [ ]:
# Verify column order
print("\n=== COLUMN ORDER (first 10) ===")
for i, col in enumerate(ikg_df.columns[:10], 1):
    print(f"{i:2}. {col}")
print(f"\n✓ Column 5 = {ikg_df.columns[4]} (labels)")
print(f"✓ Column 6 = {ikg_df.columns[5]} (epic)")
print(f"✓ Column 7 = {ikg_df.columns[6]} (weight)")
print(f"\nTotal columns: {len(ikg_df.columns)}")

In [ ]:
# Create Excel
wb = Workbook()
wb.remove(wb.active)

for name, df in [('IKG', ikg_df), ('SWAT', swat_df)]:
    sheet = wb.create_sheet(name)
    for r_idx, r in enumerate(dataframe_to_rows(df, index=False, header=True)):
        sheet.append(r)
        if r_idx == 0:
            for cell in sheet[1]:
                cell.font = Font(bold=True, color='FFFFFF')
                cell.fill = PatternFill(start_color='366092', end_color='366092', fill_type='solid')
                cell.alignment = Alignment(horizontal='center')

wb.save('gitlab_issues_ikg_swat.xlsx')
print("✓ Excel created")

In [ ]:
# Greenplum
db_password = getpass.getpass("Greenplum Password: ")
conn = psycopg2.connect(
    host=GREENPLUM_HOST, port=GREENPLUM_PORT, database=GREENPLUM_DB,
    user=GREENPLUM_USER, password=db_password
)
conn.autocommit = False
print("✓ Connected")

In [ ]:
def create_table(schema, table, df, drop=True, if_not_exists=False):
    cur = conn.cursor()
    try:
        if drop:
            cur.execute(f"DROP TABLE IF EXISTS {schema}.{table}")
            print(f"  Dropped {schema}.{table}")
        
        cols = [f"{c} {'BIGINT' if df[c].dtype == 'int64' else 'DOUBLE PRECISION' if df[c].dtype == 'float64' else 'BOOLEAN' if df[c].dtype == 'bool' else 'TEXT'}" 
                for c in df.columns]
        
        sql = f"CREATE TABLE {'IF NOT EXISTS ' if if_not_exists else ''}{schema}.{table} ({', '.join(cols)}) DISTRIBUTED RANDOMLY"
        cur.execute(sql)
        conn.commit()
        print(f"  ✓ {table}")
    except Exception as e:
        if if_not_exists and 'exists' in str(e).lower():
            print(f"  ✓ {table} exists")
            conn.commit()
        else:
            conn.rollback()
            raise
    finally:
        cur.close()

def insert_copy(schema, table, df):
    cur = conn.cursor()
    try:
        buf = StringIO()
        df.to_csv(buf, index=False, header=False, sep='\t', na_rep='\\N')
        buf.seek(0)
        sql = f"COPY {schema}.{table} ({', '.join(df.columns)}) FROM STDIN WITH (FORMAT CSV, DELIMITER E'\\t', NULL '\\N')"
        cur.copy_expert(sql, buf)
        conn.commit()
        print(f"  ✓ {table}: {len(df)} rows")
    except Exception as e:
        conn.rollback()
        raise
    finally:
        cur.close()

print("✓ DB functions ready")

In [ ]:
# Main tables (drop & create)
print("\nMAIN tables (drop & create)...")
create_table(GREENPLUM_SCHEMA, OUTPUT_TABLE1, ikg_df, drop=True, if_not_exists=False)
create_table(GREENPLUM_SCHEMA, OUTPUT_TABLE2, swat_df, drop=True, if_not_exists=False)

In [ ]:
# Archive tables (if not exists)
print("\nARCHIVE tables (if not exists)...")
create_table(GREENPLUM_SCHEMA, ARCHIVE_TABLE1, ikg_df, drop=False, if_not_exists=True)
create_table(GREENPLUM_SCHEMA, ARCHIVE_TABLE2, swat_df, drop=False, if_not_exists=True)

In [ ]:
# Insert main
print("\nInserting MAIN (COPY)...")
insert_copy(GREENPLUM_SCHEMA, OUTPUT_TABLE1, ikg_df)
insert_copy(GREENPLUM_SCHEMA, OUTPUT_TABLE2, swat_df)

In [ ]:
# Insert archive
print("\nInserting ARCHIVE (COPY)...")
insert_copy(GREENPLUM_SCHEMA, ARCHIVE_TABLE1, ikg_df)
insert_copy(GREENPLUM_SCHEMA, ARCHIVE_TABLE2, swat_df)

In [ ]:
conn.close()
print("\n" + "="*60)
print("✓ COMPLETE")
print(f"IKG: {len(ikg_df)} | SWAT: {len(swat_df)}")
print(f"Columns: {len(ikg_df.columns)}")
print(f"Order: title → labels → epic → weight → description...")
print(f"📝 Log: {log_filename}")
print("="*60)